In [1]:
import torch
import random
import math
import testdata
from multicor_fa._em import fit_EM_iter

In [2]:
%load_ext autoreload

In [3]:
%autoreload 2

Complete data case

In [4]:
def run_test(m=3, n=5000, n_iter=1000, data='complete'):
    metrics = {
        'WWt_corr': [],
        'LLt_corr': [],
        'Phi_corr': [],
        'Sigma_corr': [] # WW^T + LL^T + Phi
    }
    
    for i in range(n_iter):
        if (i+1) % (0.2*n_iter) == 0:
            print(f"{i+1} simulations completed")

        # generate dataset parameters
        if m == 'dynamic': 
            m = random.randint(3,5)
        params = testdata.simulate_data_params(m=m, n=n, private_var=True)

        # generate data
        Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=True)
        Y = Y.T
            
        if data=='missing_disjoint':
            # For default n=5000, n_m=50+75+25=150 
            i1 = math.floor(params['n']*0.2) # i1=1000 for default n=5000
            i2 = i1 + math.floor(params['n']*0.01) # i2=1050 for default n=5000
            Y[i1:i2, :params['p'][0]] = float('nan')
            i3 = i2 + math.floor(params['n']*0.01) # i3=1100 for default n=5000
            i4 = i3 + math.floor(params['n']*0.015) # i4=1175 for default n=5000
            Y[i3:i4, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
            i5 = i4 + math.floor(params['n']*0.01) # i5=1225 for default n=5000
            i6 = i5 + math.floor(params['n']*0.005) # i6=1250 for default n=5000
            Y[i5:i6, params['p'][0]+params['p'][1]:] = float('nan')
    
        elif data=='missing_overlap':
            # For default n=5000, n_m=50+50+125-25-25=175
            i1 = math.floor(params['n']*0.2) # i1=1000 for default n=5000
            i2 = i1 + math.floor(params['n']*0.01) # i2=1050 for default n=5000
            Y[i1:i2, :params['p'][0]] = float('nan')
            i3 = i2 - math.floor(params['n']*0.005) # i3=1025 for default n=5000
            i4 = i2 + math.floor(params['n']*0.025) # i4=1175 for default n=5000
            Y[i3:i4, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
            i5 = i4 - math.floor(params['n']*0.005) # i5=1150 for default n=5000
            i6 = i5 + math.floor(params['n']*0.025) # i6=1275 for default n=5000
            Y[i5:i6, params['p'][0]+params['p'][1]:] = float('nan')

        W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=True)
        Sigma_hat = Y.nan_to_num().T @ Y.nan_to_num() / Y.shape[0]

        # ground truths
        WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
        LL = (torch.block_diag(*L_true)) @ (torch.block_diag(*L_true).T)
        P = torch.cat([Phi.flatten() for Phi in Phi_true])
        Sigma_true = torch.cat([
            torch.flatten(
                (W_true[i] @ W_true[i].T) + 
                (L_true[i] @ L_true[i].T) + 
                Phi_true[i]
            ) 
            for i in range(len(W_true))
        ])
    
        # run EM
        W_new, L_new, Phi_new, _, _ = fit_EM_iter(
            Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000, impute=(data!='complete')
        )
        WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
        LL_test = (torch.block_diag(*L_new)) @ (torch.block_diag(*L_new).T)
        P_test = torch.cat([Phi.flatten() for Phi in Phi_new])
        Sigma_test = torch.cat([
            torch.flatten(
                (W_new[i] @ W_new[i].T) + 
                (L_new[i] @ L_new[i].T) +
                Phi_new[i]
            ) 
            for i in range(len(W_new))
        ])
        
        # get metrics
        metrics['WWt_corr'].append(torch.corrcoef(
            torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
        )[0,1].item())
        metrics['LLt_corr'].append(torch.corrcoef(
            torch.stack([LL.flatten(), LL_test.flatten()], dim=0)
        )[0,1].item())
        metrics['Phi_corr'].append(torch.corrcoef(
            torch.stack([P, P_test], dim=0)
        )[0,1].item())
        metrics['Sigma_corr'].append(torch.corrcoef(
            torch.stack([Sigma_true, Sigma_test], dim=0)
        )[0,1].item())

    print('\nFinal Metrics:')
    for k, v in metrics.items():
        print(' ', k)
        v = torch.tensor(v)
        print(f"\tMean: {round(torch.mean(v).item(), 4)}")
        print(f"\tMin: {round(torch.min(v).item(), 4)}")
        print(f"\tMax: {round(torch.max(v).item(), 4)}")

Complete data

In [9]:
run_test(n_iter=100)

20 simulations completed
40 simulations completed
60 simulations completed
80 simulations completed
100 simulations completed

Final Metrics:
  WWt_corr
	Mean: 0.9929
	Min: 0.9337
	Max: 0.9964
  LLt_corr
	Mean: 0.995
	Min: 0.9271
	Max: 0.9988
  Phi_corr
	Mean: 0.9951
	Min: 0.9749
	Max: 0.9992
  Sigma_corr
	Mean: 0.9983
	Min: 0.9978
	Max: 0.9988


Missing data, with each sample missing maximum 1 mode

In [10]:
run_test(data='missing_disjoint', n_iter=100)

20 simulations completed
40 simulations completed
60 simulations completed
80 simulations completed
100 simulations completed

Final Metrics:
  WWt_corr
	Mean: 0.9939
	Min: 0.9593
	Max: 0.9961
  LLt_corr
	Mean: 0.9953
	Min: 0.9351
	Max: 0.9979
  Phi_corr
	Mean: 0.9855
	Min: 0.9607
	Max: 0.9948
  Sigma_corr
	Mean: 0.9981
	Min: 0.997
	Max: 0.9987


Missing data, with each sample missing up to 2/3 modes

In [11]:
run_test(data='missing_overlap', n_iter=100)

20 simulations completed
40 simulations completed
60 simulations completed
80 simulations completed
100 simulations completed

Final Metrics:
  WWt_corr
	Mean: 0.9917
	Min: 0.958
	Max: 0.9959
  LLt_corr
	Mean: 0.9924
	Min: 0.9445
	Max: 0.9975
  Phi_corr
	Mean: 0.9792
	Min: 0.9316
	Max: 0.9949
  Sigma_corr
	Mean: 0.9977
	Min: 0.9962
	Max: 0.9984
